# Gold layer — feature engineering

This notebook builds the Gold layer for the Kitsune SYN DoS anomaly detection
project. It reads from `kitsune_project.silver_layer.syn_dos_clean` and
produces model-ready datasets.

## Scope
This notebook covers two responsibilities:

1. **Feature selection** — reduce the 115 raw features to a smaller set,
   based on the discriminant power scores and correlation groups identified
   in `03_EDA`.
2. **Train/validation/test split** — split the dataset respecting
   chronological order. Random splitting is not valid here: the attack in
   this capture occurs within a specific time window rather than being
   distributed uniformly across the capture, so the split strategy depends
   on where that window falls relative to `row_id`.



## Locating the attack window and its internal distribution

Before designing the train/cv/test split, we need to know exactly where the
attack falls within the timeline (`row_id` range) and how it is distributed
inside that window. Random splitting is not valid here, since the attack is
concentrated in a specific time range rather than spread uniformly across
the capture.

Result: attacks are heavily front-loaded (74% fall within the first 3
deciles), tapering off toward the end of the capture. This shapes the split
strategy in the next section.

In [0]:
from pyspark.sql import functions as F

silver_df = spark.table("kitsune_project.silver_layer.syn_dos_clean")

# Get the first row_id, last row_id, and total count of attack rows
attack_row_id_range = (
    silver_df
    .filter(F.col("label") == 1)
    .agg(
        F.min("row_id").alias("first_attack_row_id"),
        F.max("row_id").alias("last_attack_row_id"),
        F.count("*").alias("attack_row_count")
    )
    .collect()[0]
)

# Store the boundaries in variables for later use
window_start = attack_row_id_range["first_attack_row_id"]
window_end = attack_row_id_range["last_attack_row_id"]
window_span = window_end - window_start

total_row_count = silver_df.count()

# Express the attack window as a percentage of the full timeline
first_pct = window_start / total_row_count * 100
last_pct = window_end / total_row_count * 100

print(f"Total rows: {total_row_count}")
print(f"First attack row_id: {window_start}")
print(f"Last attack row_id: {window_end}")
print(f"Attack row count: {attack_row_id_range['attack_row_count']}")
print(f"Attack window covers {first_pct:.2f}% to {last_pct:.2f}% of the capture")

In [0]:
# Keep only attack rows that fall inside the window we just found
attack_window_df = silver_df.filter(
    (F.col("row_id") >= window_start) & (F.col("label") == 1)
)

# Split the window into 10 equal chunks (deciles) and count attacks per chunk
attack_distribution = (
    attack_window_df
    .withColumn(
        "decile",
        ((F.col("row_id") - window_start) / window_span * 10).cast("int")
    )
    .groupBy("decile")
    .agg(F.count("*").alias("attack_count"))
    .orderBy("decile")
)

attack_distribution.show()

## Building the train, cv, and test splits

With the attack window and its internal distribution known, the splits are
defined as contiguous, chronologically ordered blocks based on percentiles
inside the attack window (`row_id`-based, no random shuffling, since a
random split would break temporal causality and let the model train on
future data).

Cutoffs used: 20% and 40% of the attack window span. This yields
train = deciles 0-1, cv = deciles 2-3, test = deciles 4-10 of the attack
window.

The 20%/40% cutoffs give cv and
test more attack examples,
which produces more statistically stable evaluation metrics. Train still
retains a large number of attack examples (3,403), and class imbalance
within train, so the reduced
training volume is not a meaningful trade-off here.

The sanity check below confirms no split lost rows, and that every split
contains attack examples.

In [0]:
# compute the cut points for train, cv, and test based on percentiles inside the attack window
train_cv_cutoff = window_start + int(window_span * 0.2)  
cv_test_cutoff = window_start + int(window_span * 0.4)   

print(f"Train ends at row_id: {train_cv_cutoff}")
print(f"CV ends at row_id: {cv_test_cutoff}")

In [0]:
# Cell: apply the cut points to build train, cv, and test sets
train_df = silver_df.filter(F.col("row_id") < train_cv_cutoff)
cv_df = silver_df.filter(
    (F.col("row_id") >= train_cv_cutoff) & (F.col("row_id") < cv_test_cutoff)
)
test_df = silver_df.filter(F.col("row_id") >= cv_test_cutoff)

# Sanity check: confirm each split has attack examples 
for name, df in [("train", train_df), ("cv", cv_df), ("test", test_df)]:
    total = df.count()
    attacks = df.filter(F.col("label") == 1).count()
    print(f"{name}: {total} rows, {attacks} attacks")